# Computing transition matrix with LP

In [ ]:
from scipy.spatial.distance import pdist, squareform
import numpy as np
import pandas as pd

In [ ]:
# For n vectors of d dimensions
vectors = np.random.rand(10, 100).T  # 1000 vectors, 100 dimensions


In [ ]:
vectors.shape

In [ ]:
# Compute condensed distance matrix (upper triangle only)
distances_condensed = pdist(vectors, metric='euclidean')

# Convert to full square matrix if needed
distances_full = squareform(distances_condensed)

In [ ]:
distances_full.shape
pd.DataFrame(distances_full)

In [ ]:
# For Euclidean distance
diff = vectors[:, np.newaxis, :] - vectors[np.newaxis, :, :]  # (n, n, d)
distances = np.linalg.norm(diff, axis=2)

In [ ]:
pd.DataFrame(distances)

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances

distances = pairwise_distances(vectors, metric='euclidean')

In [ ]:
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt

def solve_transition_lp(
    D,
    P_prev,
    lam=1.0,
    solver="HIGHs",
    warm_start=True,
    verbose=False
):
    """
    Solve:
        max <D, P> - lam * ||P - P_prev||_1
    subject to:
        P >= 0
        row sums = 1
        col sums = 1
        diag(P) = 0
    """

    d = D.shape[0]

    # Decision variables
    P = cp.Variable((d, d))
    Z = cp.Variable((d, d))

    constraints = []

    # Non-negativity
    constraints += [P >= 0]

    # Row and column stochasticity
    constraints += [cp.sum(P, axis=1) == 1]
    constraints += [cp.sum(P, axis=0) == 1]

    # Zero diagonal
    constraints += [cp.diag(P) == 0]

    # L1 regularization constraints
    constraints += [Z >= P - P_prev]
    constraints += [Z >= -(P - P_prev)]

    # Objective
    objective = cp.Maximize(cp.sum(cp.multiply(D, P)) - lam * cp.sum(Z))

    problem = cp.Problem(objective, constraints)

    problem.solve(
        solver=cp.HIGHS,
        warm_start=warm_start,
        verbose=verbose,
    )

    if problem.status not in ["optimal", "optimal_inaccurate"]:
        raise RuntimeError(f"LP did not solve: {problem.status}")

    return P.value



In [ ]:
d = 1000

# Distance matrix (example: Euclidean on a line)
x = np.linspace(0, 1, d)
D = np.abs(x[:, None] - x[None, :])

# Create a distance matrix based on angular measured between d angles from 0 to 2pi
np.random.seed(0)
angles = np.sort(np.random.uniform(0, 2 * np.pi, d))
print(angles)
abs_angular_dist = np.abs(angles[:, None] - angles[None, :])
D = np.minimum(abs_angular_dist, 2 * np.pi - abs_angular_dist)

# Initial doubly stochastic matrix (uniform, zero diagonal)
P0 = np.ones((d, d)) / (d - 1)
np.fill_diagonal(P0, 0)

# Solve
P1 = solve_transition_lp(D, P0, lam=2, verbose=False)
P1

In [ ]:
plt.imshow(P1, cmap='viridis')
plt.colorbar()
plt.title('Optimized Transition Matrix P1')
plt.show()

In [ ]:
P_matrices = [None] * 7
lambdas = [0.01, 0.05, 0.1, 1, 2, 5, 10]
for j in range(len(lambdas)):
    P_matrices[j] = solve_transition_lp(D, P0, lam=lambdas[j], verbose=False)


In [ ]:
# Plot the transition matrices for different lambda values in a grid
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flatten()):
    if i < len(lambdas):
        im = ax.imshow(P_matrices[i], cmap='viridis')
        ax.set_title(f'Lambda = {lambdas[i]}')
        fig.colorbar(im, ax=ax)
    else:
        ax.axis('off')
plt.tight_layout()
plt.show()